In [1]:
! pip install pandas pydub requests

Active code page: 1252
Defaulting to user installation because normal site-packages is not writeable


In [2]:
! pip install librosa soundfile requests

Active code page: 1252
Defaulting to user installation because normal site-packages is not writeable


In [3]:
!pip install librosa soundfile requests pandas tqdm

Active code page: 1252
Defaulting to user installation because normal site-packages is not writeable


In [4]:
import pandas as pd
import requests
import json
import os
import re
from tqdm import tqdm

# ── EDIT THIS PATH ──
METADATA_CSV = r"C:\Users\satya\Downloads\JoshTalks_ASR_Assignment\task\data\raw_inputs\your_dataset.csv"
OUTPUT_CSV   = r"C:\Users\satya\Downloads\JoshTalks_ASR_Assignment\task\Q2_Disfluency_Detection\disfluency_output.csv"
CLIPPED_DIR  = r"C:\Users\satya\Downloads\JoshTalks_ASR_Assignment\task\Q2_Disfluency_Detection\clipped_audio"

os.makedirs(CLIPPED_DIR, exist_ok=True)

# Hindi + English disfluency tokens
DISFLUENCY_TOKENS = {
    "filler": [
        "हम्म", "हम्मम", "ह्म्म", "उम्म", "उम", "हाँ", "हां", "जी",
        "आं", "अं", "ओह", "अच्छा", "मतलब", "यानी",
        "hmm", "umm", "uh", "uhh", "um", "err", "ah", "ahh"
    ],
    "repetition": [],       # detected by pattern
    "prolongation": [],     # detected by pattern
    "hesitation": [
        "वो", "वो वो", "मैं मैं", "है है", "और और"
    ]
}

print("Disfluency keywords loaded.")
print("Clipped audio folder:", CLIPPED_DIR)

Disfluency keywords loaded.
Clipped audio folder: C:\Users\satya\Downloads\JoshTalks_ASR_Assignment\task\Q2_Disfluency_Detection\clipped_audio


In [6]:
import os

folder = r"C:\Users\satya\Downloads\JoshTalks_ASR_Assignment\task\data\raw_inputs"

print("Files in raw_inputs folder:")
for f in os.listdir(folder):
    print(" →", f)

Files in raw_inputs folder:
 → FT Data - data.csv
 → FT_Data_with_fixed_urls.csv
 → Question 4 - Task.csv
 → Speech Disfluencies List - Sheet1.csv
 → Unique Words Data - Sheet1.csv


In [7]:
METADATA_CSV    = r"C:\Users\satya\Downloads\JoshTalks_ASR_Assignment\task\data\raw_inputs\FT_Data_with_fixed_urls.csv"
DISFLUENCY_LIST = r"C:\Users\satya\Downloads\JoshTalks_ASR_Assignment\task\data\raw_inputs\Speech Disfluencies List - Sheet1.csv"
OUTPUT_CSV      = r"C:\Users\satya\Downloads\JoshTalks_ASR_Assignment\task\Q2_Disfluency_Detection\disfluency_output.csv"
CLIPPED_DIR     = r"C:\Users\satya\Downloads\JoshTalks_ASR_Assignment\task\Q2_Disfluency_Detection\clipped_audio"

os.makedirs(CLIPPED_DIR, exist_ok=True)
print("Paths set!")

Paths set!


In [8]:
df = pd.read_csv(METADATA_CSV)
print("Dataset columns:", df.columns.tolist())
print("Total rows:", len(df))

dis_df = pd.read_csv(DISFLUENCY_LIST)
print("\nDisfluency list columns:", dis_df.columns.tolist())
print("Total keywords:", len(dis_df))
dis_df.head(10)

Dataset columns: ['user_id', 'recording_id', 'language', 'duration', 'rec_url_gcp', 'transcription_url_gcp', 'metadata_url_gcp', 'fixed_audio_url', 'fixed_trans_url']
Total rows: 104

Disfluency list columns: ['Filled Pause', 'Repetition', 'False Start', 'Prolongation', 'Self-Correction']
Total keywords: 67


,Filled Pause,Repetition,False Start,Prolongation,Self-Correction
0,अं,मैं-मैं,जा—,अच्छ्छ्छा,कल—
1,अँ,वो-वो,कर—,हम्म्म,नहीं—
2,उम्,ये-ये,ले—,आाा,परसों—
3,NaN,जी-जी,कह—,अरे रे रे,माफ़—
4,हम्,उह उह,वो तो,अह,नहीं तो हां तो ये
5,हम्म,दो दो,तब तक हां,नहीं नी नी नी,हम्म जी नहीं
6,उम्म,हम्म हम्म,आप अ,अ अ अ अ,पहले तो इतना मतलब
7,अह,वो तो वो तो,बोव,रुकोओओ,ये तो बाकी तो
8,ओके,हाँ-हाँ-हाँ-हाँ,अच्छ,नाआआ,मो मा में महाराष्ट्र
9,अ अ,हाँ यस सर – हाँ यस सर,बचन,हांआआ,हां ये भी है ये है है ऐ


In [9]:
import re

# Load all keywords from all 5 columns
all_keywords = []
for col in dis_df.columns:
    keywords = dis_df[col].dropna().str.strip().str.lower().tolist()
    all_keywords.extend(keywords)

all_keywords = [k for k in all_keywords if k]
print("Total keywords loaded:", len(all_keywords))
print("Sample:", all_keywords[:10])

# Column-wise keyword map for type detection
keyword_map = {}
for col in dis_df.columns:
    keyword_map[col] = dis_df[col].dropna().str.strip().str.lower().tolist()

Total keywords loaded: 214
Sample: ['अं', 'अँ', 'उम्', 'हम्', 'हम्म', 'उम्म', 'अह', 'ओके', 'अ अ', 'ह ह ह']


In [10]:
# Get disfluency tokens from the CSV
disfluency_tokens = dis_df.iloc[:, 0].dropna().str.strip().tolist()
print("Total tokens loaded:", len(disfluency_tokens))
print("Sample:", disfluency_tokens[:10])

Total tokens loaded: 33
Sample: ['अं', 'अँ', 'उम्', 'हम्', 'हम्म', 'उम्म', 'अह', 'ओके', 'अ अ', 'ह ह ह']


In [11]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print("Cleaned columns:", df.columns.tolist())
df.head(3)

Cleaned columns: ['user_id', 'recording_id', 'language', 'duration', 'rec_url_gcp', 'transcription_url_gcp', 'metadata_url_gcp', 'fixed_audio_url', 'fixed_trans_url']


,user_id,recording_id,language,duration,rec_url_gcp,transcription_url_gcp,metadata_url_gcp,fixed_audio_url,fixed_trans_url
0,245746,825780,hi,443,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/upload_goai/967...,https://storage.googleapis.com/upload_goai/967...
1,291038,825727,hi,443,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/upload_goai/967...,https://storage.googleapis.com/upload_goai/967...
2,246004,988596,hi,475,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/upload_goai/114...,https://storage.googleapis.com/upload_goai/114...


In [12]:
def detect_disfluency_type(text):
    text_lower = str(text).lower().strip()
    detected = []

    # Check against loaded keyword list
    for token in disfluency_tokens:
        if str(token).lower() in text_lower:
            detected.append("filler")
            break

    # Repetition pattern (वो वो, मैं मैं)
    import re
    if re.search(r'(\b\w+\b)\s+\1', text_lower, re.UNICODE):
        detected.append("repetition")

    # Prolongation (आआआ, sooooo)
    if re.search(r'(.)\1{2,}', text_lower, re.UNICODE):
        detected.append("prolongation")

    return detected if detected else None

In [13]:
all_disfluencies = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    try:
        recording_id = str(row['recording_id'])
        user_id      = str(row['user_id'])
        trans_url    = str(row['fixed_trans_url'])
        audio_url    = str(row['fixed_audio_url'])

        resp = requests.get(trans_url, timeout=15)
        if resp.status_code != 200:
            continue

        data = resp.json()

        # Handle both list and dict format
        if isinstance(data, dict):
            segments = data.get('segments', data.get('utterances', []))
        elif isinstance(data, list):
            segments = data
        else:
            continue

        for seg in segments:
            text       = seg.get('text', seg.get('transcript', ''))
            start_time = seg.get('start', seg.get('start_time', 0))
            end_time   = seg.get('end',   seg.get('end_time',   0))

            if not text:
                continue

            disfluency_types = detect_disfluency_type(text)

            if disfluency_types:
                for dtype in disfluency_types:
                    all_disfluencies.append({
                        'recording_id'         : recording_id,
                        'user_id'              : user_id,
                        'segment_start'        : start_time,
                        'segment_end'          : end_time,
                        'disfluency_type'      : dtype,
                        'transcription_snippet': text.strip(),
                        'audio_url'            : audio_url,
                        'audio_segment_path'   : ''
                    })

    except Exception as e:
        print(f"Error {recording_id}: {e}")
        continue

disfluency_df = pd.DataFrame(all_disfluencies)
print(f"\nTotal disfluency segments found: {len(disfluency_df)}")
disfluency_df.head(10)

100%|██████████| 104/104 [00:46<00:00,  2.21it/s]



Total disfluency segments found: 1690


,recording_id,user_id,segment_start,segment_end,disfluency_type,transcription_snippet,audio_url,audio_segment_path
0,825780,245746,52.70,66.83,repetition,हां तो फिर वहां जो दिन भर तो दिन में तो खोजने ...,https://storage.googleapis.com/upload_goai/967...,
1,825780,245746,89.06,103.19,repetition,डर लगता है लेकिन बहुत सारे थे तो पता सब अपना अ...,https://storage.googleapis.com/upload_goai/967...,
2,825780,245746,130.25,144.83,filler,हम्म हम लोग की तो सुबह होती है की मम्मी उठाने ...,https://storage.googleapis.com/upload_goai/967...,
3,825780,245746,146.18,158.21,repetition,अराम से हां आराम से सो रहे थे हम और पता है जब ...,https://storage.googleapis.com/upload_goai/967...,
4,825780,245746,160.67,173.24,filler,बहुत अजीब सा अनुभव था क्योंकि वो पता है उठा उठ...,https://storage.googleapis.com/upload_goai/967...,
5,825780,245746,178.34,190.85,repetition,"हां बाहरी आ गया तो, वहां के जो थे ना वो अपने भ...",https://storage.googleapis.com/upload_goai/967...,
6,825780,245746,257.66,264.08,repetition,पता है लेकिन जब लोग घूमने जाते हैं तो लाइट वगै...,https://storage.googleapis.com/upload_goai/967...,
7,825727,291038,67.19,74.48,repetition,जी डर डर डर लगा,https://storage.googleapis.com/upload_goai/967...,
8,825727,291038,101.36,104.18,filler,जी जी जी आह,https://storage.googleapis.com/upload_goai/967...,
9,825727,291038,143.15,148.10,repetition,हा उठा के सीधे आप सोई हुए थे आप सोई हुए थे आपक...,https://storage.googleapis.com/upload_goai/967...,


In [14]:
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
disfluency_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print("Saved:", OUTPUT_CSV)
print("Total rows:", len(disfluency_df))
print("\nBy type:")
print(disfluency_df['disfluency_type'].value_counts())

Saved: C:\Users\satya\Downloads\JoshTalks_ASR_Assignment\task\Q2_Disfluency_Detection\disfluency_output.csv
Total rows: 1690

By type:
disfluency_type
filler          980
repetition      701
prolongation      9
Name: count, dtype: int64
